In [0]:
%run "../config/00_config"

In [0]:
%run "../config/tables/ecommerce_rastreamento_entregas_config"

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = build_adls_options(
    storage_account_name=ADLS_STORAGE_ACCOUNT_NAME,
    client_id=ADLS_CLIENT_ID,
    tenant_id=ADLS_TENANT_ID,
    client_secret=ADLS_CLIENT_SECRET,
)

print("Configuração da carga:")
print(f"Arquivo origem: {SOURCE_FILE}")
print(f"Caminho origem: {SOURCE_PATH}")
print(f"Tabela destino: {TARGET_FULL_TABLE}")
print(f"Modo de escrita: {WRITE_MODE}")

In [0]:
df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS,
)

df_source = cast_all_columns_to_string(df_source)
df_source = add_load_timestamp(df_source)

print("Arquivo fonte lido com sucesso.")
print("Todas as colunas foram convertidas para string.")

In [0]:
df_source.printSchema()

In [0]:
source_overview = get_dataframe_overview(df_source)

total_linhas_origem = source_overview["total_linhas"]
total_colunas_origem = source_overview["total_colunas"]

if total_linhas_origem == 0:
    raise ValueError(f"O arquivo {SOURCE_FILE} foi lido sem registros.")

print("Resumo da origem:")
print(f"Total de linhas: {total_linhas_origem}")
print(f"Total de colunas: {total_colunas_origem}")
print(f"Colunas encontradas: {source_overview['colunas']}")

In [0]:
if VALIDATE_EXPECTED_COLUMNS:
    columns_validation = validate_required_columns(
        df=df_source,
        expected_columns=EXPECTED_COLUMNS,
    )

    print(columns_validation["message"])

    if columns_validation["unexpected_columns"]:
        print(f"Colunas adicionais encontradas: {columns_validation['unexpected_columns']}")
else:
    print("Validação de colunas esperadas desativada no config.")


if VALIDATE_KEY_COLUMNS:
    key_validation = validate_key_columns(
        df=df_source,
        key_columns=KEY_COLUMNS,
    )

    print(key_validation["message"])
    print(f"Colunas de chave validadas: {key_validation['key_columns']}")
else:
    print("Validação de chave desativada no config.")

In [0]:
print(f"Atenção: a tabela {TARGET_FULL_TABLE} será gravada com mode='{WRITE_MODE}'.")

write_sql_table(
    df=df_source,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=TARGET_FULL_TABLE,
    mode=WRITE_MODE,
    sql_port=SQL_PORT,
)

print(f"Tabela gravada com sucesso no SQL Server: {TARGET_FULL_TABLE}")

In [0]:
df_target = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=TARGET_FULL_TABLE,
    sql_port=SQL_PORT,
)

count_validation = compare_row_counts(
    source_df=df_source,
    target_df=df_target,
)

print(count_validation["message"])
print(f"Total de registros na origem: {count_validation['source_count']}")
print(f"Total de registros no destino: {count_validation['target_count']}")